# Prefix Alarm Monitor

This notebook is the main experiment walkthrough for the prefix alarm monitor. The reusable implementation and default experiment config live in `prefix_alarm_monitor_lib.py`, so the notebook can stay focused on the data flow, rule comparison, and results.

Current experiment setup:
- Train only on mixed-root files, where `0 < root_y < 1`.
- Validate and test only on pure-root files, where `root_y = 0` or `root_y = 1`.
- Reconstruct each prefix along the node's ancestor path.
- Keep a truncated rolling window of the latest `64` steps.
- Sample prefixes every `8` steps, plus the final step of each node segment.
- Skip prefixes with `step_idx <= 8`, because those early prefixes have too little trajectory information.
- Learn the raw prefix score with weighted sibling Bradley-Terry comparisons, but keep a small row-level anchor so the score keeps a stable absolute scale.
- Compare three leaf-level monitor rules: `calibrated_leaf_low_fp`, `cusum_leaf_low_fp`, and `gated_leaf_low_fp`.


## Important: Why We Skip The First 8 Steps

The previous `root alarm` metric looked at `node_id == "0"`, which is the shared root segment before the first fork. In this dataset, that root segment is usually steps `1` through `8`.

Those prefixes are too early to be a serious alarm signal. A step-1 alarm mostly reflects weak task/repo priors and the first action, not a meaningful trajectory pattern.

So this notebook sets:

```python
MIN_STEP = 9
```

That means training samples, pairwise sibling examples, saved predictions, and metrics all ignore `step_idx <= 8`.


## 1. Load CSV Files And Inspect Root Labels

Each `*.steps.csv` file is one SWE-bench instance. We use the first row's `y` value as the root solvability label for that tree.

Root groups:

- `pure0`: root `y = 0`, meaning no successful leaves under the root.
- `pure1`: root `y = 1`, meaning all leaves under the root succeeded.
- `mixed`: root `y` is between `0` and `1`, which is most useful for training because it contains branching disagreement.


In [25]:
import prefix_alarm_monitor_lib as monitor_lib

globals().update(monitor_lib.notebook_globals())

print("data dir:", DATA_DIR)
print("output dir:", OUTPUT_DIR)
print("target leaf recall:", TARGET_LEAF_RECALL)
print("rule threshold max candidates:", RULE_THRESHOLD_MAX_CANDIDATES)


data dir: /mnt/d/Filez/Desktop/Research/raed/test/data
output dir: /mnt/d/Filez/Desktop/Research/raed/test/monitor_results/leaf_low_fp_monitor
target leaf recall: 0.85
rule threshold max candidates: 48


In [26]:
# load_all_instances parses every CSV, groups rows by node_id, and reconstructs
# ancestor-path rows for each node. Those path rows are what later become prefix windows.
instances = tam.load_all_instances(DATA_DIR)

root_kind_counts = Counter(instance.root_kind for instance in instances.values())
root_y_values = [instance.root_y for instance in instances.values()]

print("num instances:", len(instances))
print("root kind counts:", dict(root_kind_counts))
print("root y min / mean / max:", min(root_y_values), np.mean(root_y_values), max(root_y_values))

root_y_table = pd.Series(root_y_values, name="root_y").value_counts().sort_index().reset_index()
root_y_table.columns = ["root_y", "num_files"]
display(root_y_table.head(12))
display(root_y_table.tail(12))


num instances: 707
root kind counts: {'mixed': 253, 'pure0': 232, 'pure1': 222}
root y min / mean / max: 0.0 0.5156692008486563 1.0


,root_y,num_files
0,0.000000,232
1,0.007812,3
2,0.015625,4
3,0.023438,4
4,0.031250,4
5,0.039062,4
6,0.046875,2
7,0.054688,3
8,0.070312,3
9,0.078125,1


,root_y,num_files
92,0.906250,5
93,0.914062,1
94,0.921875,2
95,0.929688,7
96,0.937500,2
97,0.953125,4
98,0.960938,4
99,0.968750,7
100,0.976562,7
101,0.984375,9


## 2. Split Files

The split is done by file, not by row. This avoids having prefixes from the same instance appear in both training and evaluation.

All mixed files go into training. Pure files are split into validation and test with class balance preserved as much as possible.


In [27]:
splits = tam.stratified_pure_split(
    instances=instances,
    pure_val_ratio=PURE_VAL_RATIO,
    seed=SEED,
)

split_table = pd.DataFrame({
    "split": ["train_mixed", "val_pure", "test_pure"],
    "num_files": [len(splits.train_mixed), len(splits.val_pure), len(splits.test_pure)],
    "pure0_files": [
        sum(instances[x].root_kind == "pure0" for x in splits.train_mixed),
        sum(instances[x].root_kind == "pure0" for x in splits.val_pure),
        sum(instances[x].root_kind == "pure0" for x in splits.test_pure),
    ],
    "pure1_files": [
        sum(instances[x].root_kind == "pure1" for x in splits.train_mixed),
        sum(instances[x].root_kind == "pure1" for x in splits.val_pure),
        sum(instances[x].root_kind == "pure1" for x in splits.test_pure),
    ],
    "mixed_files": [
        sum(instances[x].root_kind == "mixed" for x in splits.train_mixed),
        sum(instances[x].root_kind == "mixed" for x in splits.val_pure),
        sum(instances[x].root_kind == "mixed" for x in splits.test_pure),
    ],
})

display(split_table)


,split,num_files,pure0_files,pure1_files,mixed_files
0,train_mixed,253,0,0,253
1,val_pure,227,116,111,0
2,test_pure,227,116,111,0


## 3. Build Prefix Samples

`build_samples` converts tree rows into prefix-level training examples.

For each sampled prefix we keep:

- `instance_id`: the SWE-bench task id.
- `node_id`: the tree node.
- `step_idx`: the prefix length along the trajectory.
- `target_y`: the subtree success rate for this node.
- `feature_vector`: features extracted from the latest 64-step window.

Because `MIN_STEP = 9`, prefixes from steps `1` through `8` are not created at all.


In [28]:
featurizer = tam.PrefixFeaturizer()

bundle = tam.build_samples(
    instances=instances,
    featurizer=featurizer,
    split_spec=splits,
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    min_step=MIN_STEP,
    min_pair_gap=MIN_PAIR_GAP,
)

print("num features:", len(bundle.feature_names))
print("train samples:", len(bundle.train_samples))
print("val samples:", len(bundle.val_samples))
print("test samples:", len(bundle.test_samples))
print("pairwise sibling examples:", bundle.pair_diffs.shape[0])

all_sample_steps = [s.step_idx for s in bundle.train_samples + bundle.val_samples + bundle.test_samples]
print("min sampled step:", min(all_sample_steps))

example = bundle.train_samples[0]
print("example sample")
print("  instance_id:", example.instance_id)
print("  node_id:", example.node_id)
print("  step_idx:", example.step_idx)
print("  target_y:", example.target_y)
print("  feature shape:", example.feature_vector.shape)


num features: 148
train samples: 75036
val samples: 38222
test samples: 39136
pairwise sibling examples: 7565
min sampled step: 9
example sample
  instance_id: astropy__astropy-12907
  node_id: 0.0
  step_idx: 9
  target_y: 0.84375
  feature shape: (148,)


## 4. Convert Samples To Matrices

The model is linear, so each `PrefixSample` becomes one row in a feature matrix.

The scaler is fit only on mixed training samples. Validation and test data are transformed with that training scaler to avoid leakage.


In [29]:
x_train_raw, y_train = tam.samples_to_arrays(bundle.train_samples)
x_val_raw, y_val_soft = tam.samples_to_arrays(bundle.val_samples)
x_test_raw, y_test_soft = tam.samples_to_arrays(bundle.test_samples)

scaler = tam.Standardizer().fit(x_train_raw)

x_train = scaler.transform(x_train_raw)
x_val = scaler.transform(x_val_raw)
x_test = scaler.transform(x_test_raw)

# Pairwise diffs are x_left - x_right. After standardization, the mean cancels,
# so we only divide by the scaler's standard deviation.
pair_diffs = bundle.pair_diffs / scaler.scale_

# For pure files, pure0 is the failure/alarm class and pure1 is the success class.
y_val_fail = np.array([1 if s.root_kind == "pure0" else 0 for s in bundle.val_samples])
y_test_fail = np.array([1 if s.root_kind == "pure0" else 0 for s in bundle.test_samples])

print("x_train:", x_train.shape)
print("x_val:", x_val.shape)
print("x_test:", x_test.shape)
print("pair_diffs:", pair_diffs.shape)


x_train: (75036, 148)
x_val: (38222, 148)
x_test: (39136, 148)
pair_diffs: (7565, 148)


## 5. Train The Monitor

The core monitor score is still the linear prefix score from the idea:

```python
s(v) = x_v^T theta
```

We now keep the weighted sibling Bradley-Terry objective as the main signal, but add a small row-level BCE anchor (`ROW_WEIGHT = 0.2`) so the learned score keeps a usable absolute scale. That makes calibration and uncertainty estimates much more stable than the pure-pairwise version.


In [30]:
model = tam.LinearAlarmModel(
    num_features=x_train.shape[1],
    learning_rate=LEARNING_RATE,
    epochs=EPOCHS,
    row_weight=ROW_WEIGHT,
    pairwise_weight=PAIRWISE_WEIGHT,
    l2=L2,
)

model.fit(
    x_train=x_train,
    y_train=y_train,
    x_val_pure=x_val,
    y_val_pure_fail=y_val_fail,
    pair_diffs=pair_diffs,
    pair_labels=bundle.pair_labels,
    pair_weights=bundle.pair_weights,
)

print("best epoch:", model.best_epoch)
print("best validation AUC:", model.best_val_auc)


best epoch: 13
best validation AUC: 0.8810663040584866


## 6. Evaluate The Leaf-Level Monitor

We now keep only **leaf-level** evaluation and compare three monitoring rules:

- `calibrated_leaf_low_fp`
- `cusum_leaf_low_fp`
- `gated_leaf_low_fp`

The default final rule is still `calibrated_leaf_low_fp`, while `cusum_leaf_low_fp` is the sequential-monitoring baseline that accumulates risk over time along each leaf path.


In [31]:
train_raw_scores = model.predict_score(x_train)
val_raw_scores = model.predict_score(x_val)
test_raw_scores = model.predict_score(x_test)

calibrator = tam.LogisticCalibrator(
    learning_rate=CALIBRATION_LEARNING_RATE,
    epochs=CALIBRATION_EPOCHS,
    l2=CALIBRATION_L2,
).fit(train_raw_scores, y_train)

train_p_success = calibrator.predict_success_prob(train_raw_scores)
val_p_success = calibrator.predict_success_prob(val_raw_scores)
test_p_success = calibrator.predict_success_prob(test_raw_scores)
val_fail_scores = 1.0 - val_p_success
test_fail_scores = 1.0 - test_p_success

parameter_covariance = model.parameter_covariance(
    pair_diffs,
    bundle.pair_weights,
    row_features=x_train,
)
train_score_se = model.score_standard_error(x_train, parameter_covariance)
val_score_se = model.score_standard_error(x_val, parameter_covariance)
test_score_se = model.score_standard_error(x_test, parameter_covariance)

train_lcb_scores = train_raw_scores - CONFIDENCE_Z * train_score_se
val_lcb_scores = val_raw_scores - CONFIDENCE_Z * val_score_se
test_lcb_scores = test_raw_scores - CONFIDENCE_Z * test_score_se

node_final_score_map = tam.build_node_final_score_map(
    instances=instances,
    featurizer=featurizer,
    window_size=WINDOW_SIZE,
    scaler=scaler,
    model=model,
)
val_parent_scores = tam.compute_parent_scores(bundle.val_samples, node_final_score_map)
test_parent_scores = tam.compute_parent_scores(bundle.test_samples, node_final_score_map)
val_score_drops = val_raw_scores - val_parent_scores
test_score_drops = test_raw_scores - test_parent_scores

val_trajectory_index = tam.build_trajectory_index(bundle.val_samples, instances)
test_trajectory_index = tam.build_trajectory_index(bundle.test_samples, instances)

calibrated_leaf_low_fp_info = tam.choose_leaf_aware_success_threshold(
    y_true_fail=y_val_fail,
    p_success=val_p_success,
    trajectory_index=val_trajectory_index,
    target_leaf_recall=TARGET_LEAF_RECALL,
    max_candidates=RULE_THRESHOLD_MAX_CANDIDATES,
)
final_success_threshold = calibrated_leaf_low_fp_info["success_threshold"]
val_calibrated_leaf_low_fp_pred = (val_p_success < final_success_threshold).astype(np.int64)
test_calibrated_leaf_low_fp_pred = (test_p_success < final_success_threshold).astype(np.int64)

cusum_threshold_info = tam.choose_leaf_aware_cusum_thresholds(
    fail_scores=val_fail_scores,
    trajectory_index=val_trajectory_index,
    target_leaf_recall=TARGET_LEAF_RECALL,
    max_candidates=RULE_THRESHOLD_MAX_CANDIDATES,
)
val_cusum_metrics = tam.trajectory_cusum_metrics_from_index(
    val_fail_scores,
    val_trajectory_index,
    cusum_threshold_info["cusum_drift"],
    cusum_threshold_info["cusum_threshold"],
)
test_cusum_metrics = tam.trajectory_cusum_metrics_from_index(
    test_fail_scores,
    test_trajectory_index,
    cusum_threshold_info["cusum_drift"],
    cusum_threshold_info["cusum_threshold"],
)

gated_threshold_info = tam.choose_leaf_aware_composite_thresholds(
    y_true_fail=y_val_fail,
    p_success=val_p_success,
    lcb_scores=val_lcb_scores,
    score_drops=val_score_drops,
    trajectory_index=val_trajectory_index,
    success_threshold=final_success_threshold,
    target_leaf_recall=TARGET_LEAF_RECALL,
    max_candidates=RULE_THRESHOLD_MAX_CANDIDATES,
    gate_by_success=True,
)
gated_lcb_threshold = gated_threshold_info["lcb_threshold"]
gated_drop_threshold = gated_threshold_info["drop_threshold"]
val_gated_rule = tam.apply_monitor_rule(
    lcb_scores=val_lcb_scores,
    score_drops=val_score_drops,
    lcb_threshold=gated_lcb_threshold,
    drop_threshold=gated_drop_threshold,
    calibrated_success=val_p_success,
    success_threshold=final_success_threshold,
    gate_by_success=True,
)
test_gated_rule = tam.apply_monitor_rule(
    lcb_scores=test_lcb_scores,
    score_drops=test_score_drops,
    lcb_threshold=gated_lcb_threshold,
    drop_threshold=gated_drop_threshold,
    calibrated_success=test_p_success,
    success_threshold=final_success_threshold,
    gate_by_success=True,
)

final_rule_name = "calibrated_leaf_low_fp"

leaf_rule_metrics = {
    "val": {
        "calibrated_leaf_low_fp": tam.trajectory_alarm_metrics_from_index(val_calibrated_leaf_low_fp_pred, val_trajectory_index),
        "cusum_leaf_low_fp": val_cusum_metrics,
        "gated_leaf_low_fp": tam.trajectory_alarm_metrics_from_index(val_gated_rule["pred_fail"], val_trajectory_index),
    },
    "test": {
        "calibrated_leaf_low_fp": tam.trajectory_alarm_metrics_from_index(test_calibrated_leaf_low_fp_pred, test_trajectory_index),
        "cusum_leaf_low_fp": test_cusum_metrics,
        "gated_leaf_low_fp": tam.trajectory_alarm_metrics_from_index(test_gated_rule["pred_fail"], test_trajectory_index),
    },
}

val_final_leaf_alarm_metrics = leaf_rule_metrics["val"][final_rule_name]
test_final_leaf_alarm_metrics = leaf_rule_metrics["test"][final_rule_name]

leaf_rule_table = pd.DataFrame([
    {"rule": "calibrated_leaf_low_fp", "split": "val", **leaf_rule_metrics["val"]["calibrated_leaf_low_fp"]},
    {"rule": "calibrated_leaf_low_fp", "split": "test", **leaf_rule_metrics["test"]["calibrated_leaf_low_fp"]},
    {"rule": "cusum_leaf_low_fp", "split": "val", **leaf_rule_metrics["val"]["cusum_leaf_low_fp"]},
    {"rule": "cusum_leaf_low_fp", "split": "test", **leaf_rule_metrics["test"]["cusum_leaf_low_fp"]},
    {"rule": "gated_leaf_low_fp", "split": "val", **leaf_rule_metrics["val"]["gated_leaf_low_fp"]},
    {"rule": "gated_leaf_low_fp", "split": "test", **leaf_rule_metrics["test"]["gated_leaf_low_fp"]},
])

print("final rule:", final_rule_name)
print("leaf-aware final success_probability threshold:", final_success_threshold)
print("cusum drift:", cusum_threshold_info["cusum_drift"])
print("cusum threshold:", cusum_threshold_info["cusum_threshold"])
print("gated_lcb threshold:", gated_lcb_threshold)
print("gated_drop threshold:", gated_drop_threshold)
print("target leaf recall:", TARGET_LEAF_RECALL)
print("confidence_z:", CONFIDENCE_Z)

display(leaf_rule_table[[
    "rule",
    "split",
    "pure0_leaf_alarm_recall",
    "pure1_leaf_false_alarm_rate",
    "pure0_leaf_alarm_median_step",
    "pure1_leaf_false_alarm_median_step",
    "pure0_successful_warning_mean_early_pct",
    "pure0_leaf_paths",
    "pure1_leaf_paths",
]])


final rule: calibrated_leaf_low_fp
leaf-aware final success_probability threshold: 0.35120224293907587
cusum drift: 6.629363724641735e-12
cusum threshold: 4.864531821461264
gated_lcb threshold: -2561.978126725034
gated_drop threshold: 0.06414448322092206
target leaf recall: 0.85
confidence_z: 1.96


,rule,split,pure0_leaf_alarm_recall,pure1_leaf_false_alarm_rate,pure0_leaf_alarm_median_step,pure1_leaf_false_alarm_median_step,pure0_successful_warning_mean_early_pct,pure0_leaf_paths,pure1_leaf_paths
0,calibrated_leaf_low_fp,val,0.851629,0.188312,40.0,32.0,38.847506,8654.0,1232.0
1,calibrated_leaf_low_fp,test,0.843363,0.180328,40.0,24.0,38.484326,8957.0,1159.0
2,cusum_leaf_low_fp,val,0.856020,0.133117,48.0,48.0,22.576044,8654.0,1232.0
3,cusum_leaf_low_fp,test,0.821480,0.103538,49.0,41.0,22.306783,8957.0,1159.0
4,gated_leaf_low_fp,val,0.850936,0.183442,40.0,32.0,38.406116,8654.0,1232.0
5,gated_leaf_low_fp,test,0.841353,0.172563,40.0,24.0,38.048963,8957.0,1159.0


### CUSUM Alarm Backtracking

For each CUSUM-alarmed leaf path, this traces backward from the first alarm to the shortest recent contribution window whose cumulative CUSUM increments are enough to cross the alarm threshold. The reported rollback is the distance from the alarm step to that inferred risk-start step.


In [32]:
def cusum_shortest_contribution_windows(
    fail_scores,
    trajectory_index,
    cusum_drift,
    cusum_threshold,
    split_name,
):
    rows = []
    fail_scores = np.asarray(fail_scores, dtype=np.float64)

    for path_id, item in enumerate(trajectory_index):
        sample_indices = np.asarray(item["sample_indices"], dtype=np.int64)
        path_steps = np.asarray(item["path_steps"], dtype=np.int64)
        if sample_indices.size == 0:
            continue

        increments = fail_scores[sample_indices] - float(cusum_drift)
        stat = 0.0
        segment_start = 0
        running_at_alarm = np.nan
        alarm_pos = None

        for pos, increment in enumerate(increments):
            candidate = stat + float(increment)
            if candidate <= 0.0:
                stat = 0.0
                segment_start = pos + 1
            else:
                stat = candidate

            if stat >= float(cusum_threshold):
                alarm_pos = pos
                running_at_alarm = stat
                break

        if alarm_pos is None:
            continue

        suffix_sum = 0.0
        risk_start_pos = segment_start
        for start_pos in range(alarm_pos, segment_start - 1, -1):
            suffix_sum += float(increments[start_pos])
            if suffix_sum >= float(cusum_threshold) - 1e-12:
                risk_start_pos = start_pos
                break

        rows.append(
            {
                "split": split_name,
                "path_id": path_id,
                "root_kind": item["root_kind"],
                "final_step": int(item["final_step"]),
                "risk_start_step": int(path_steps[risk_start_pos]),
                "alarm_step": int(path_steps[alarm_pos]),
                "rollback_steps": int(path_steps[alarm_pos] - path_steps[risk_start_pos]),
                "rollback_sampled_prefixes": int(alarm_pos - risk_start_pos),
                "window_sampled_prefixes": int(alarm_pos - risk_start_pos + 1),
                "alarm_cusum": float(running_at_alarm),
                "window_contribution": float(suffix_sum),
            }
        )

    return pd.DataFrame(rows)

val_cusum_backtrack = cusum_shortest_contribution_windows(
    val_fail_scores,
    val_trajectory_index,
    cusum_threshold_info["cusum_drift"],
    cusum_threshold_info["cusum_threshold"],
    "val",
)
test_cusum_backtrack = cusum_shortest_contribution_windows(
    test_fail_scores,
    test_trajectory_index,
    cusum_threshold_info["cusum_drift"],
    cusum_threshold_info["cusum_threshold"],
    "test",
)
cusum_backtrack_windows = pd.concat(
    [val_cusum_backtrack, test_cusum_backtrack],
    ignore_index=True,
)

if cusum_backtrack_windows.empty:
    print("No CUSUM alarms were found, so there are no windows to backtrack.")
else:
    summary_by_kind = (
        cusum_backtrack_windows
        .groupby(["split", "root_kind"], as_index=False)
        .agg(
            alarmed_leaf_paths=("path_id", "count"),
            mean_rollback_steps=("rollback_steps", "mean"),
            median_rollback_steps=("rollback_steps", "median"),
            mean_rollback_sampled_prefixes=("rollback_sampled_prefixes", "mean"),
            median_rollback_sampled_prefixes=("rollback_sampled_prefixes", "median"),
            mean_window_sampled_prefixes=("window_sampled_prefixes", "mean"),
            median_alarm_step=("alarm_step", "median"),
            median_risk_start_step=("risk_start_step", "median"),
        )
    )
    summary_overall = (
        cusum_backtrack_windows
        .groupby("split", as_index=False)
        .agg(
            alarmed_leaf_paths=("path_id", "count"),
            mean_rollback_steps=("rollback_steps", "mean"),
            median_rollback_steps=("rollback_steps", "median"),
            mean_rollback_sampled_prefixes=("rollback_sampled_prefixes", "mean"),
            median_rollback_sampled_prefixes=("rollback_sampled_prefixes", "median"),
            mean_window_sampled_prefixes=("window_sampled_prefixes", "mean"),
            median_alarm_step=("alarm_step", "median"),
            median_risk_start_step=("risk_start_step", "median"),
        )
    )
    summary_overall.insert(1, "root_kind", "all")

    cusum_backtrack_summary = pd.concat(
        [summary_overall, summary_by_kind],
        ignore_index=True,
    )

    display(cusum_backtrack_summary[[
        "split",
        "root_kind",
        "alarmed_leaf_paths",
        "mean_rollback_steps",
        "median_rollback_steps",
        "mean_rollback_sampled_prefixes",
        "median_rollback_sampled_prefixes",
        "mean_window_sampled_prefixes",
        "median_risk_start_step",
        "median_alarm_step",
    ]])

    display(
        cusum_backtrack_windows
        .sort_values(["split", "rollback_steps"], ascending=[True, False])
        .head(10)[[
            "split",
            "root_kind",
            "path_id",
            "risk_start_step",
            "alarm_step",
            "rollback_steps",
            "rollback_sampled_prefixes",
            "window_sampled_prefixes",
            "alarm_cusum",
            "window_contribution",
        ]]
    )


,split,root_kind,alarmed_leaf_paths,mean_rollback_steps,median_rollback_steps,mean_rollback_sampled_prefixes,median_rollback_sampled_prefixes,mean_window_sampled_prefixes,median_risk_start_step,median_alarm_step
0,test,all,7478,34.574886,32.0,8.576892,8.0,9.576892,16.0,49.0
1,val,all,7572,34.213286,32.0,8.509641,8.0,9.509641,16.0,48.0
2,test,pure0,7358,34.643245,32.0,8.591193,8.0,9.591193,16.0,49.0
3,test,pure1,120,30.383333,32.0,7.700000,8.0,8.700000,9.0,41.0
4,val,pure0,7408,34.246625,32.0,8.519033,8.0,9.519033,16.0,48.0
5,val,pure1,164,32.707317,32.0,8.085366,8.0,9.085366,9.0,48.0


,split,root_kind,path_id,risk_start_step,alarm_step,rollback_steps,rollback_sampled_prefixes,window_sampled_prefixes,alarm_cusum,window_contribution
7914,test,pure0,590,9,64,55,13,14,4.864689,4.864689
8795,test,pure0,1879,9,64,55,13,14,4.900989,4.900989
8885,test,pure0,2095,9,64,55,13,14,4.867487,4.867487
10233,test,pure0,3871,9,64,55,13,14,4.898993,4.898993
10296,test,pure0,4028,9,64,55,13,14,4.955951,4.955951
10297,test,pure0,4029,9,64,55,13,14,4.971628,4.971628
11574,test,pure0,5514,9,64,55,13,14,4.911938,4.911938
11604,test,pure0,5611,9,64,55,13,14,4.920173,4.920173
11605,test,pure0,5613,9,64,55,13,14,4.874221,4.874221
12041,test,pure0,6273,9,64,55,13,14,4.874678,4.874678


## 7. No-Confidence Ablation

This reruns the same experiment after dropping every feature whose name contains `confidence`, so we can see how much the monitor depends on that signal.


In [33]:
NO_CONFIDENCE_OUTPUT_DIR = ROOT / "monitor_results" / "no_confidence_leaf_low_fp_monitor"

no_conf_args = argparse.Namespace(
    data_dir=str(DATA_DIR),
    output_dir=str(NO_CONFIDENCE_OUTPUT_DIR),
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    min_step=MIN_STEP,
    pure_val_ratio=PURE_VAL_RATIO,
    seed=SEED,
    learning_rate=LEARNING_RATE,
    epochs=EPOCHS,
    row_weight=ROW_WEIGHT,
    pairwise_weight=PAIRWISE_WEIGHT,
    l2=L2,
    min_pair_gap=MIN_PAIR_GAP,
    calibration_learning_rate=CALIBRATION_LEARNING_RATE,
    calibration_epochs=CALIBRATION_EPOCHS,
    calibration_l2=CALIBRATION_L2,
    confidence_z=CONFIDENCE_Z,
    target_leaf_recall=TARGET_LEAF_RECALL,
    rule_threshold_max_candidates=RULE_THRESHOLD_MAX_CANDIDATES,
    drop_confidence_features=True,
)
no_conf_summary = tam.run_experiment(no_conf_args)

comparison_rows = []
rule_order = [
    "calibrated_leaf_low_fp",
    "cusum_leaf_low_fp",
    "gated_leaf_low_fp",
]
for setup_name, metrics_by_rule in [
    ("full_features", leaf_rule_metrics["test"]),
    ("no_confidence", no_conf_summary["metrics"]["leaf_rule_comparison"]["test"]),
]:
    for rule_name in rule_order:
        row = {"setup": setup_name, "rule": rule_name}
        row.update(metrics_by_rule[rule_name])
        comparison_rows.append(row)

no_conf_comparison_table = pd.DataFrame(comparison_rows)
display(no_conf_comparison_table[[
    "setup",
    "rule",
    "pure0_leaf_alarm_recall",
    "pure1_leaf_false_alarm_rate",
    "pure0_leaf_alarm_median_step",
    "pure0_successful_warning_mean_early_pct",
]])

print("no-confidence output dir:", NO_CONFIDENCE_OUTPUT_DIR)
print("dropped confidence features:", len(no_conf_summary["feature_filter"]["dropped_features"]))


,setup,rule,pure0_leaf_alarm_recall,pure1_leaf_false_alarm_rate,pure0_leaf_alarm_median_step,pure0_successful_warning_mean_early_pct
0,full_features,calibrated_leaf_low_fp,0.843363,0.180328,40.0,38.484326
1,full_features,cusum_leaf_low_fp,0.821480,0.103538,49.0,22.306783
2,full_features,gated_leaf_low_fp,0.841353,0.172563,40.0,38.048963
3,no_confidence,calibrated_leaf_low_fp,0.851401,0.201898,41.0,36.592119
4,no_confidence,cusum_leaf_low_fp,0.824606,0.104400,49.0,20.535780
5,no_confidence,gated_leaf_low_fp,0.843809,0.192407,41.0,35.291525


no-confidence output dir: /mnt/d/Filez/Desktop/Research/raed/test/monitor_results/no_confidence_leaf_low_fp_monitor
dropped confidence features: 11


## 8. Feature Weights

This is a linear model, so feature weights are directly inspectable. Positive weights increase `p_success`; negative weights decrease it.

These are standardized-feature weights, so they are useful for direction and rough magnitude, not causal interpretation.

We also report a per-feature Wald significance computed from the observed Fisher information of the trained model (the same `parameter_covariance` used downstream for score standard errors): `std_err` is `sqrt(diag(cov))`, `z = weight / std_err`, and `p_value = erfc(|z| / sqrt(2))`. Rows are sorted from most to least significant (largest `|z|` / smallest `p_value` first). With L2 regularization the Fisher is ridge-penalized, so treat these p-values as a heuristic ranking, not a strict frequentist test.


In [34]:
cov = model.parameter_covariance(
    pair_diffs=pair_diffs,
    pair_weights=bundle.pair_weights,
    row_features=x_train,
)
std_err = np.sqrt(np.clip(np.diag(cov), 0.0, None))
with np.errstate(divide="ignore", invalid="ignore"):
    z_scores = np.where(std_err > 0, model.weights / std_err, 0.0)
p_values = np.array([math.erfc(abs(z) / math.sqrt(2.0)) for z in z_scores])

weights = pd.DataFrame({
    "feature": bundle.feature_names,
    "weight": model.weights,
    "std_err": std_err,
    "z": z_scores,
    "p_value": p_values,
})
weights["abs_z"] = np.abs(weights["z"])

ranked = weights.sort_values("abs_z", ascending=False).head(25)
display(ranked[["feature", "weight", "std_err", "z", "p_value"]])


,feature,weight,std_err,z,p_value
27,cur__target_is_abs_like,-0.158604,0.021851,-7.258266,3.920825e-13
146,win__command_switch_ratio,-0.246875,0.038032,-6.491245,8.512985e-11
107,win__target_missing__mean,-0.282948,0.043670,-6.479288,9.215635e-11
98,win__confidence__max,0.142750,0.022043,6.476022,9.417233e-11
99,win__confidence__std,0.216846,0.039874,5.438292,5.379387e-08
43,win__step_wall_s__min,0.127861,0.024834,5.148548,2.625102e-07
96,win__confidence__mean,0.212005,0.042005,5.047091,4.485880e-07
78,win__repeat_cmd_score_recent__mean,-0.276284,0.055220,-5.003320,5.635134e-07
145,win__consecutive_same_target_ratio,-0.182244,0.039885,-4.569189,4.896161e-06
68,win__output_len__max,-0.149005,0.035747,-4.168364,3.067942e-05


## 9. Save Results

This writes the main leaf-only baseline run, including:

- the leaf-aware success threshold used by the default low-false-positive monitor,
- the CUSUM thresholds for the sequential-monitoring baseline,
- the gated leaf-monitor thresholds for comparison,
- leaf-trajectory rule comparison metrics for the three kept rules,
- and per-prefix prediction flags for the default final rule.


In [35]:
summary = {
    "config": {
        "data_dir": str(DATA_DIR),
        "window_size": WINDOW_SIZE,
        "stride": STRIDE,
        "min_step": MIN_STEP,
        "pure_val_ratio": PURE_VAL_RATIO,
        "seed": SEED,
        "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS,
        "row_weight": ROW_WEIGHT,
        "pairwise_weight": PAIRWISE_WEIGHT,
        "l2": L2,
        "min_pair_gap": MIN_PAIR_GAP,
        "calibration_learning_rate": CALIBRATION_LEARNING_RATE,
        "calibration_epochs": CALIBRATION_EPOCHS,
        "calibration_l2": CALIBRATION_L2,
        "confidence_z": CONFIDENCE_Z,
        "target_leaf_recall": TARGET_LEAF_RECALL,
        "rule_threshold_max_candidates": RULE_THRESHOLD_MAX_CANDIDATES,
        "drop_confidence_features": False,
    },
    "counts": {
        "num_instances": len(instances),
        "mixed_train_files": len(bundle.splits.train_mixed),
        "val_pure_files": len(bundle.splits.val_pure),
        "test_pure_files": len(bundle.splits.test_pure),
        "train_samples": len(bundle.train_samples),
        "val_samples": len(bundle.val_samples),
        "test_samples": len(bundle.test_samples),
        "pairwise_examples": int(bundle.pair_diffs.shape[0]),
        "num_features_before_filter": len(bundle.feature_names),
        "num_features_after_filter": len(bundle.feature_names),
    },
    "feature_filter": {
        "dropped_features": [],
    },
    "sample_summary": {
        "train": tam.summarize_samples(bundle.train_samples),
        "val": tam.summarize_samples(bundle.val_samples),
        "test": tam.summarize_samples(bundle.test_samples),
    },
    "thresholds": {
        "calibrated_leaf_low_fp": calibrated_leaf_low_fp_info,
        "success_probability_threshold": final_success_threshold,
        "cusum_leaf_low_fp": cusum_threshold_info,
        "gated_leaf_low_fp": gated_threshold_info,
        "confidence_z": CONFIDENCE_Z,
    },
    "model": {
        "best_epoch": model.best_epoch,
        "best_val_auc": model.best_val_auc,
        "bias": model.bias,
        "calibrator": calibrator.to_dict(),
    },
    "metrics": {
        "train_soft_target": {
            "row_bce": tam.soft_bce_loss(y_train, train_p_success),
            "pairwise_bce": tam.pairwise_bce_loss(
                bundle.pair_labels,
                sigmoid(pair_diffs @ model.weights),
                bundle.pair_weights,
            ) if pair_diffs.shape[0] > 0 else 0.0,
            "y_mean": float(y_train.mean()),
            "pred_mean": float(train_p_success.mean()),
        },
        "leaf_rule_comparison": leaf_rule_metrics,
        "final_rule": {
            "name": final_rule_name,
            "formula": "low_calibrated_success",
            "leaf_alarm": {
                "val": val_final_leaf_alarm_metrics,
                "test": test_final_leaf_alarm_metrics,
            },
        },
    },
}

(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
(OUTPUT_DIR / "splits.json").write_text(
    json.dumps(
        {
            "train_mixed": bundle.splits.train_mixed,
            "val_pure": bundle.splits.val_pure,
            "test_pure": bundle.splits.test_pure,
        },
        indent=2,
    ),
    encoding="utf-8",
)
(OUTPUT_DIR / "training_history.json").write_text(json.dumps(model.history, indent=2), encoding="utf-8")
(OUTPUT_DIR / "scaler.json").write_text(json.dumps(scaler.to_dict()), encoding="utf-8")
(OUTPUT_DIR / "calibrator.json").write_text(json.dumps(calibrator.to_dict(), indent=2), encoding="utf-8")

tam.save_predictions_csv(
    OUTPUT_DIR / "val_predictions.csv",
    bundle.val_samples,
    val_p_success,
    val_raw_scores,
    val_lcb_scores,
    val_parent_scores,
    val_score_drops,
    val_calibrated_leaf_low_fp_pred,
    np.zeros_like(val_calibrated_leaf_low_fp_pred),
    np.zeros_like(val_calibrated_leaf_low_fp_pred),
    val_calibrated_leaf_low_fp_pred,
)
tam.save_predictions_csv(
    OUTPUT_DIR / "test_predictions.csv",
    bundle.test_samples,
    test_p_success,
    test_raw_scores,
    test_lcb_scores,
    test_parent_scores,
    test_score_drops,
    test_calibrated_leaf_low_fp_pred,
    np.zeros_like(test_calibrated_leaf_low_fp_pred),
    np.zeros_like(test_calibrated_leaf_low_fp_pred),
    test_calibrated_leaf_low_fp_pred,
)
tam.save_feature_weights(OUTPUT_DIR / "feature_weights.csv", bundle.feature_names, model.weights)

top_weights = sorted(
    zip(bundle.feature_names, model.weights), key=lambda item: abs(item[1]), reverse=True
)[:20]
(OUTPUT_DIR / "top_weights.json").write_text(
    json.dumps(
        [{"feature": feature, "weight": float(weight)} for feature, weight in top_weights],
        indent=2,
    ),
    encoding="utf-8",
)

print("saved to:", OUTPUT_DIR)


saved to: /mnt/d/Filez/Desktop/Research/raed/test/monitor_results/leaf_low_fp_monitor
